# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [7]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Find the repository root from the notebook's current working directory.
repo_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data" / "raw" / "content_refresh_anonymized.csv").exists()
)
os.chdir(repo_root)

DATA_PATH = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Repository root:", repo_root)
print("Broj stranica:", len(df))
print("Broj kolona:", len(df.columns))

Repository root: c:\Users\lukau\FlyRank_intership
Broj stranica: 30000
Broj kolona: 44


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [8]:
df["declining_proxy"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Proxy target:", "declining_proxy")
print(df["declining_proxy"].value_counts())
print(
    "Udeo stranica sa padom:",
    round(df["declining_proxy"].mean(), 3)
)

Proxy target: declining_proxy
declining_proxy
1    16262
0    13738
Name: count, dtype: int64
Udeo stranica sa padom: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [9]:
top_k = 50

declining_rate = df["declining_proxy"].mean()

print(f"Baseline rate svih stranica: {declining_rate:.3f}")
print(f"Očekivani broj declining stranica među {top_k}: "
      f"{round(declining_rate * top_k)}")

Baseline rate svih stranica: 0.542
Očekivani broj declining stranica među 50: 27


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
lane = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "sessions_90d",
        "avg_position",
        "ctr",
        "word_count",
        "trend_direction",
        "declining_proxy",
    ]
].copy()

display(lane.head(10))
print("Shape:", lane.shape)
print("Jedan red = jedna content stranica")

,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,sessions_90d,avg_position,ctr,word_count,trend_direction,declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,17,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,9,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,78,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,145,44.0,0.13,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,5,8.5,0.03,3080.0,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,1,7.0,0.00,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,28,21.2,0.06,NaN,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,68,46.0,0.09,3807.0,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,3,4.9,0.16,NaN,down,1


Shape: (30000, 12)
Jedan red = jedna content stranica


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [11]:
df["rule_score"] = (
    (df["days_since_last_update"] >= 180)
    & (df["impressions_90d"] >= 500)
).astype(int) * df["impressions_90d"]

top_50_rule = df.sort_values(
    "rule_score",
    ascending=False
).head(50)

print(
    "Rule Precision@50:",
    round(top_50_rule["declining_proxy"].mean(), 3)
)

Rule Precision@50: 0.74


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.